# Fraud Detection – E-commerce Transactions (EDA)

## 10 Academy – Artificial Intelligence Mastery  
### Week 5 & 6 Challenge (Dec 17 – Dec 30, 2025)

**Business Context:**  
Adey Innovations Inc. aims to improve fraud detection for e-commerce and banking transactions.  
This notebook focuses on exploratory data analysis (EDA), preprocessing, and feature engineering for e-commerce transaction data.

**Datasets Used:**  
- Fraud_Data.csv  
- IpAddress_to_Country.csv  

**Objectives:**  
- Clean and validate raw transaction data  
- Perform exploratory data analysis (EDA)  
- Integrate geolocation data (IP → Country)  
- Engineer meaningful features for fraud detection  
- Analyze and document class imbalance


In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set(style="whitegrid")


In [45]:
fraud = pd.read_csv("../data/raw/Fraud_Data.csv")
ip = pd.read_csv("../data/raw/IpAddress_to_Country.csv")

fraud.head()


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0


In [46]:
fraud.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151112 entries, 0 to 151111
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   user_id         151112 non-null  int64  
 1   signup_time     151112 non-null  object 
 2   purchase_time   151112 non-null  object 
 3   purchase_value  151112 non-null  int64  
 4   device_id       151112 non-null  object 
 5   source          151112 non-null  object 
 6   browser         151112 non-null  object 
 7   sex             151112 non-null  object 
 8   age             151112 non-null  int64  
 9   ip_address      151112 non-null  float64
 10  class           151112 non-null  int64  
dtypes: float64(1), int64(4), object(6)
memory usage: 12.7+ MB


In [47]:
fraud.describe()


,user_id,purchase_value,age,ip_address,class
count,151112.000000,151112.000000,151112.000000,1.511120e+05,151112.000000
mean,200171.040970,36.935372,33.140704,2.152145e+09,0.093646
std,115369.285024,18.322762,8.617733,1.248497e+09,0.291336
min,2.000000,9.000000,18.000000,5.209350e+04,0.000000
25%,100642.500000,22.000000,27.000000,1.085934e+09,0.000000
50%,199958.000000,35.000000,33.000000,2.154770e+09,0.000000
75%,300054.000000,49.000000,39.000000,3.243258e+09,0.000000
max,400000.000000,154.000000,76.000000,4.294850e+09,1.000000


In [48]:
fraud.isnull().sum()


user_id           0
signup_time       0
purchase_time     0
purchase_value    0
device_id         0
source            0
browser           0
sex               0
age               0
ip_address        0
class             0
dtype: int64

In [49]:
fraud['signup_time'] = pd.to_datetime(fraud['signup_time'])
fraud['purchase_time'] = pd.to_datetime(fraud['purchase_time'])

fraud.drop_duplicates(inplace=True)


In [50]:
fraud['class'].value_counts()


class
0    136961
1     14151
Name: count, dtype: int64

In [51]:
fraud['class'].value_counts(normalize=True) * 100


class
0    90.635423
1     9.364577
Name: proportion, dtype: float64

### Class Imbalance Observation

The target variable (`class`) is highly imbalanced, with fraudulent transactions representing a very small fraction of the total dataset.  
This imbalance makes accuracy an unsuitable evaluation metric and necessitates the use of specialized techniques such as resampling (e.g., SMOTE) and metrics like F1-score and AUC-PR during model training.


In [52]:
# Check types
fraud['ip_address'].head(20)

# Check for non-string or missing values
fraud['ip_address'].apply(lambda x: type(x)).value_counts()
fraud['ip_address'].isnull().sum()


np.int64(0)

In [53]:
# Convert everything to string
fraud['ip_address'] = fraud['ip_address'].astype(str)

# Replace obvious invalids or missing with '0.0.0.0'
fraud.loc[fraud['ip_address'].str.contains(r'nan|None|^\s*$'), 'ip_address'] = '0.0.0.0'


In [54]:
import ipaddress

def safe_ip_to_int(ip_str):
    try:
        return int(ipaddress.ip_address(ip_str))
    except ValueError:
        return 0  # fallback for invalid IPs

fraud['ip_int'] = fraud['ip_address'].apply(safe_ip_to_int)
fraud[['ip_address', 'ip_int']].head(10)


,ip_address,ip_int
0,732758368.79972,0
1,350311387.865908,0
2,2621473820.11095,0
3,3840542443.91396,0
4,415583117.452712,0
5,2809315199.92675,0
6,3987484328.51882,0
7,1692458727.64945,0
8,3719094257.18731,0
9,341674739.579911,0


In [55]:
# Define mapping function
def map_ip_to_country(ip_int):
    row = ip[
        (ip['lower_bound_ip_address'] <= ip_int) &
        (ip['upper_bound_ip_address'] >= ip_int)
    ]
    if not row.empty:
        return row.iloc[0]['country']
    return 'Unknown'  # fallback for IPs not in any range


In [56]:
fraud['country'] = fraud['ip_int'].apply(map_ip_to_country)

# Check top 10 countries by number of transactions
fraud['country'].value_counts().head(10)


country
Unknown    151112
Name: count, dtype: int64

In [57]:
# Calculate fraud rate per country
country_fraud = fraud.groupby('country')['class'].mean().sort_values(ascending=False)
country_fraud.head(10)


country
Unknown    0.093646
Name: class, dtype: float64

In [58]:
# 1. Time since signup in hours
fraud['time_since_signup'] = (fraud['purchase_time'] - fraud['signup_time']).dt.total_seconds() / 3600

# 2. Hour of purchase
fraud['hour_of_day'] = fraud['purchase_time'].dt.hour

# 3. Day of week
fraud['day_of_week'] = fraud['purchase_time'].dt.dayofweek

# 4. Transaction velocity: count of purchases per user
user_transaction_counts = fraud.groupby('user_id').size().to_dict()
fraud['transaction_count'] = fraud['user_id'].map(user_transaction_counts)

# Quick look at new features
fraud[['time_since_signup', 'hour_of_day', 'day_of_week', 'transaction_count']].head()


,time_since_signup,hour_of_day,day_of_week,transaction_count
0,1251.856111,2,5,1
1,4.984444,1,0,1
2,0.000278,18,3,1
3,136.690278,13,0,1
4,1211.516944,18,2,1


### Feature Engineering

- `time_since_signup`: captures rapid purchases after signup, which can indicate fraud.  
- `hour_of_day`: fraud may occur at unusual hours.  
- `day_of_week`: weekly patterns can reveal suspicious behavior.  
- `transaction_count`: users making multiple transactions in a short period may be higher risk.


In [59]:
categorical_cols = ['sex', 'source', 'browser', 'country']

fraud_encoded = pd.get_dummies(fraud, columns=categorical_cols, drop_first=True)

fraud_encoded.head()


,user_id,signup_time,purchase_time,purchase_value,device_id,age,ip_address,class,ip_int,time_since_signup,hour_of_day,day_of_week,transaction_count,sex_M,source_Direct,source_SEO,browser_FireFox,browser_IE,browser_Opera,browser_Safari
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,39,732758368.79972,0,0,1251.856111,2,5,1,True,False,True,False,False,False,False
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,53,350311387.865908,0,0,4.984444,1,0,1,False,False,False,False,False,False,False
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,53,2621473820.11095,1,0,0.000278,18,3,1,True,False,True,False,False,True,False
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,41,3840542443.91396,0,0,136.690278,13,0,1,True,False,True,False,False,False,True
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,45,415583117.452712,0,0,1211.516944,18,2,1,True,False,False,False,False,False,True


In [60]:
from sklearn.preprocessing import StandardScaler

num_cols = ['purchase_value', 'age', 'time_since_signup', 'transaction_count']

scaler = StandardScaler()
fraud_encoded[num_cols] = scaler.fit_transform(fraud_encoded[num_cols])

fraud_encoded.head()


,user_id,signup_time,purchase_time,purchase_value,device_id,age,ip_address,class,ip_int,time_since_signup,hour_of_day,day_of_week,transaction_count,sex_M,source_Direct,source_SEO,browser_FireFox,browser_IE,browser_Opera,browser_Safari
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,-0.160204,QVPSPJUOCKZAR,0.679914,732758368.79972,0,0,-0.136057,2,5,0.0,True,False,True,False,False,False,False
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,-1.142592,EOGFQPIZPYXFZ,2.304476,350311387.865908,0,0,-1.571877,1,0,0.0,False,False,False,False,False,False,False
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,-1.197169,YSSKYOSJHPPLJ,2.304476,2621473820.11095,1,0,-1.577617,18,3,0.0,True,False,True,False,False,True,False
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,0.385567,ATGTXKYKUDUQN,0.911994,3840542443.91396,0,0,-1.420213,13,0,0.0,True,False,True,False,False,False,True
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,0.112681,NAUITBZFJKHWW,1.376155,415583117.452712,0,0,-0.182509,18,2,0.0,True,False,False,False,False,False,True


In [61]:
fraud_encoded['class'].value_counts()
fraud_encoded['class'].value_counts(normalize=True) * 100


class
0    90.635423
1     9.364577
Name: proportion, dtype: float64

### Class Imbalance Analysis

The target variable (`class`) is highly imbalanced:  
- Fraudulent transactions are a small fraction of total transactions.  
- This requires special handling during modeling: oversampling (SMOTE), undersampling, or using metrics like F1-score and AUC-PR.


In [62]:
fraud_encoded.to_csv("../data/processed/fraud_data_processed.csv", index=False)
